# Prompt Chaining with JSON Database Lookup

This notebook demonstrates a robust **Prompt Chaining** workflow that integrates with a local JSON database.

## Setup Instructions
1. Get a free Gemini API key from [Google AI Studio](https://aistudio.google.com/).
2. Open the `.env` file in the root directory and configure `GEMINI_API_KEY`.

In [17]:
import os
import re
import json
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env in the parent directory
dir_path = os.path.dirname(__file__) if "__file__" in globals() else "."
dotenv_path = os.path.abspath(os.path.join(dir_path, "..", ".env"))
load_dotenv(dotenv_path=dotenv_path)

# Retrieve the API key from environment variables
api_key = os.getenv("GEMINI_API_KEY")

if not api_key or api_key == "your_free_gemini_api_key_here":
    raise ValueError(
        "Please set your GEMINI_API_KEY in the .env file in the root of the project.\n"
        "You can get a free key from Google AI Studio: https://aistudio.google.com/"
    )

# Initialize the OpenAI client pointing to Gemini's OpenAI-compatible API endpoint
client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# We use the free gemini-2.5-flash model via the OpenAI compatible endpoint.
# If using OpenAI directly, change this to "gpt-4o-mini" or "gpt-3.5-turbo".
model_name = "gemini-3.1-flash-lite"

In [18]:
def get_completion(prompt, model=model_name):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful customer support assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def prompt_chain(initial_prompt, follow_up_prompts):
    result = get_completion(initial_prompt)
    if result is None:
        return "Initial prompt failed."

    print(f"Initial output:\n{result}\n")
    print("="*60 + "\n")

    for i, prompt in enumerate(follow_up_prompts, 1):
        full_prompt = f"{prompt}\n\nPrevious output: {result}"
        result = get_completion(full_prompt)
        if result is None:
            return f"Prompt {i} failed."
        print(f"Step {i} output:\n{result}\n")
        print("="*60 + "\n")

    return result

In [19]:
def extract_order_id(email_content):
    # Extract a 5-digit order ID from the email text
    match = re.search(r'(?:Order\s*#?\s*)?(\d{5})', email_content, re.IGNORECASE)
    return match.group(1) if match else None

def get_order_details(order_id):
    # Retrieve local path of orders_db.json
    dir_path = os.path.dirname(__file__) if "__file__" in globals() else "."
    db_path = os.path.join(dir_path, "orders_db.json")
    try:
        with open(db_path, "r") as f:
            db = json.load(f)
        return db.get(str(order_id))
    except FileNotFoundError:
        print(f"Database file '{db_path}' not found.")
        return None
    except Exception as e:
        print(f"Error reading database: {e}")
        return None

In [20]:
def run_customer_support_flow(email):
    print(f"Processing Customer Email:\n\"{email}\"\n")
    
    # Step 1: Extract Order ID
    order_id = extract_order_id(email)
    if not order_id:
        return "[ERROR] Could not extract a valid 5-digit Order ID from the email."
    
    print(f"-> Extracted Order ID: #{order_id}")
    
    # Step 2: Database Lookup
    order_details = get_order_details(order_id)
    if not order_details:
        return f"[ERROR] Database check failed: Order #{order_id} does not exist in our records."
    
    print(f"-> Database verified! Details: {order_details}\n")
    print("="*60 + "\n")
    
    # Step 3: Run Prompt Chain
    initial_prompt = f"""
    Analyze the following customer email and verified order records.
    Identify:
    1. The customer's primary complaint.
    2. The emotional sentiment.
    
    Email:
    {email}
    
    Database Record for Order #{order_id}:
    {json.dumps(order_details, indent=2)}
    """
    
    follow_up_prompts = [
        "Draft a professional, empathetic support email. Acknowledge their issue, confirm the current order status using the database facts (carrier, tracking, estimated delivery), and explain that we will refund their shipping cost.",
        "Translate the drafted email response into polite, professional Spanish."
    ]
    
    return prompt_chain(initial_prompt, follow_up_prompts)

In [21]:
# Scenario 1: Order exists in database (98765)
customer_email_success = (
    "Hi, I ordered a widget last week (Order #98765) but it hasn't arrived yet. "
    "I need it for a birthday party tomorrow! This is extremely frustrating."
)

print("--- RUNNING SCENARIO 1 (SUCCESS CASE) ---\n")
run_customer_support_flow(customer_email_success)

--- RUNNING SCENARIO 1 (SUCCESS CASE) ---

Processing Customer Email:
"Hi, I ordered a widget last week (Order #98765) but it hasn't arrived yet. I need it for a birthday party tomorrow! This is extremely frustrating."

-> Extracted Order ID: #98765
-> Database verified! Details: {'status': 'Delayed in transit', 'tracking_number': '1Z999AA10123456784', 'carrier': 'UPS', 'estimated_delivery': '2026-07-24', 'shipping_method': 'Expedited (2-day)', 'customer_name': 'John Doe'}


Initial output:
Based on the email and database records provided, here is the analysis:

**1. Primary Complaint**
The customer’s primary complaint is that their order (#98765) has not arrived by the expected date. Specifically, the customer is distressed because the item was intended for a time-sensitive event (a birthday party tomorrow) and the current "Delayed in transit" status makes it impossible for the item to arrive in time.

**2. Emotional Sentiment**
The sentiment is **frustrated and urgent**. The customer

'Aquí tienes una traducción profesional y empática al español:\n\n***\n\n**Asunto: Actualización sobre su pedido n.º 98765**\n\nEstimado/a [Nombre del cliente]:\n\nGracias por ponerse en contacto con nosotros. Lamento sinceramente saber que su pedido no ha llegado a tiempo para su próxima celebración de cumpleaños. Comprendo perfectamente lo estresante y decepcionante que resulta que un regalo con fecha límite se retrase, especialmente cuando contaba con él para una ocasión tan especial.\n\nHe revisado el estado de su envío y puedo confirmarle los siguientes detalles:\n\n*   **Transportista:** [Nombre del transportista, ej. FedEx/UPS]\n*   **Número de seguimiento:** [Número de seguimiento]\n*   **Estado actual:** Retrasado en tránsito\n*   **Entrega estimada:** [Fecha]\n\nSoy consciente de que esto no resuelve el problema inmediato de que el artículo no llegue a tiempo para su fecha límite, y por ello le pido mis más sinceras disculpas. Como gesto de nuestro compromiso con su satisfacc

In [22]:
# Scenario 2: Order does NOT exist in database (99999)
customer_email_error = (
    "Hello, where is my package for Order #99999? It's been two weeks since I paid."
)

print("\n--- RUNNING SCENARIO 2 (ERROR CASE) ---\n")
result = run_customer_support_flow(customer_email_error)
print(result)


--- RUNNING SCENARIO 2 (ERROR CASE) ---

Processing Customer Email:
"Hello, where is my package for Order #99999? It's been two weeks since I paid."

-> Extracted Order ID: #99999
[ERROR] Database check failed: Order #99999 does not exist in our records.
